# Satellite and Raster Analysis

This notebook prepares Sentinel-2 raster inputs for the GeoSoil-Cuba Jaruco/Mayabeque study area. It begins with the curated soil–vegetation analysis layer and builds a reproducible workflow for finding, screening, downloading, validating, and processing Sentinel-2 imagery.

The workflow covers:

- inspection of the curated soil–vegetation GeoPackage;
- construction of a Sentinel-2 observation inventory;
- metadata-based and AOI-specific quality screening;
- download and validation of the Scene Classification Layer (SCL);
- selection of representative acquisition dates;
- download and validation of the red (B04) and near-infrared (B08) bands;
- calculation of tile-level NDVI and creation of AOI-level NDVI rasters;
- export of available analytical tables and open figures for later reporting.

**Primary vector input:** `data/processed/vector/soil_vegetation_analysis.gpkg`

The notebook writes raster products under `data/processed/satellite/` and report-ready tables and figures under `outputs/tables/` and `outputs/figures/`. The separate NDVI analysis notebook is responsible for interpreting and visualizing the resulting time series.


## 1. Define Raster-Analysis Inputs [¶](#1-define-raster-analysis-inputs)

The raster analysis begins from the latest curated soil–vegetation analysis product generated during the AOI analysis.

The principal input is:

* `soil_vegetation_analysis.gpkg` — curated spatial dataset containing the validated soil and vegetation analysis units for the AOI.

At this stage, the objective is only to establish the input path and verify that the dataset can be loaded correctly. Satellite imagery will be introduced after the vector foundation has been confirmed.


In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd


# Project directories
PROJECT_DIR = Path("..").resolve()

# Processed data
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Satellite data

RAW_SATELLITE_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "satellite"
    / "sentinel2"
)

PROCESSED_SATELLITE_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "satellite"
)

# Curated soil–vegetation analysis GeoPackage
SOIL_VEGETATION_ANALYSIS_GPKG = (
    PROCESSED_DIR / "vector" / "soil_vegetation_analysis.gpkg"
)


# Verify input file
print("Curated soil–vegetation analysis GeoPackage:")
print(SOIL_VEGETATION_ANALYSIS_GPKG)
print("Exists:", SOIL_VEGETATION_ANALYSIS_GPKG.exists())

In [ ]:
# Load curated spatial dataset
soil_vegetation_analysis = gpd.read_file(
    SOIL_VEGETATION_ANALYSIS_GPKG
)


print("Curated soil–vegetation analysis:")
print(f"  Features: {len(soil_vegetation_analysis):,}")
print(f"  CRS:      {soil_vegetation_analysis.crs}")
print(f"  Columns:  {len(soil_vegetation_analysis.columns)}")

In [ ]:
display(soil_vegetation_analysis.head())

In [ ]:
print("Spatial geometry types:")
print(soil_vegetation_analysis.geometry.geom_type.value_counts())

print("\nMissing geometries:")
print(soil_vegetation_analysis.geometry.isna().sum())

print("\nCRS:")
print(soil_vegetation_analysis.crs)

## 2. Define Satellite-Data Requirements [¶](#2-define-satellite-data-requirements)

The satellite component will provide environmental information that can be compared with the curated soil and vegetation analysis dataset.

For the first raster analysis, the primary objective is to obtain optical satellite imagery suitable for vegetation assessment and calculation of the Normalized Difference Vegetation Index (NDVI).

The satellite dataset should provide:

* sufficient spatial resolution for the approximately 178 km² AOI;
* visible red and near-infrared (NIR) bands required for NDVI;
* acquisition dates appropriate for the study objectives;
* cloud and/or quality information allowing unsuitable pixels to be excluded;
* documented spatial reference and pixel resolution;
* data that can be processed reproducibly from the original source.

The analysis will initially focus on NDVI. Additional spectral indices or environmental variables can be added later if they provide meaningful information for the GeoSoil-Cuba objectives.

### 2.1 Raster-to-Vector Relationship [¶](#21-raster-to-vector-relationship)

The curated soil–vegetation analysis layer will provide the spatial units for extracting raster statistics.

```text
Satellite imagery
       │
       ├── Red band
       ├── NIR band
       └── Quality / cloud information
                │
                ▼
             NDVI raster
                │
                ▼
      Soil–vegetation polygons
                │
                ▼
       Raster statistics
                │
                ▼
      Soil/environment dataset
```

No raster reprojection, resampling, clipping, or index calculation is performed in this section. Those operations will be performed only after the selected satellite product and its metadata have been documented.


## 3. Sentinel-2 Observation Search and Inventory [¶](#3-sentinel-2-observation-search-and-inventory)

Sentinel-2 observations will be identified directly from the Copernicus Data Space STAC catalogue.

The search covers the GeoSoil-Cuba AOI from **1 November 2025 through 30 June 2026** and uses the Sentinel-2 Level-2A collection.

The initial search does not impose an excessively restrictive cloud threshold. Instead, all observations intersecting the AOI are retrieved so that the temporal availability can be inspected before selecting the best scenes.

The resulting scene inventory will record:

* satellite platform;
* acquisition date and time;
* Sentinel-2 product identifier;
* MGRS tile;
* reported scene cloud cover;
* spatial coverage of the AOI;
* temporal period (November–April or May–June).

This preserves the complete observation history before scenes are selected for NDVI analysis.


In [ ]:
# Copernicus Data Space STAC catalogue
STAC_URL = "https://stac.dataspace.copernicus.eu/v1"

AOI_BBOX_WGS84 = [
    -82.10,  # west
    22.92,   # south
    -81.89,  # east
    23.04    # north
]

SEARCH_START = "2025-11-01T00:00:00Z"
SEARCH_END = "2026-06-30T23:59:59Z"

print("STAC catalogue:", STAC_URL)
print("AOI bounding box:", AOI_BBOX_WGS84)
print("Search period:", SEARCH_START, "to", SEARCH_END)

In [ ]:
import pystac_client

catalog = pystac_client.Client.open(STAC_URL)

print("Connected to Copernicus Data Space STAC catalogue.")
print("Catalog title:", catalog.title)

### 3.1 Query Sentinel-2 Observations [¶](#31-query-sentinel-2-observations)

The Copernicus Data Space STAC catalogue is queried for Sentinel-2 Level-2A observations intersecting the GeoSoil-Cuba AOI between **1 March and 30 June 2026**.

At this stage, no restrictive cloud threshold is applied. The objective is to establish the complete temporal availability of Sentinel-2 observations before selecting imagery for the March–April and May–June comparison.

The resulting observations will be evaluated by acquisition date, satellite platform, MGRS tile, and reported cloud coverage.


In [ ]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=AOI_BBOX_WGS84,
    datetime=f"{SEARCH_START}/{SEARCH_END}",
    max_items=400,
)

items = list(search.items())

print(f"Sentinel-2 L2A observations found: {len(items)}")

### 3.2 Convert Results into a Scene Inventory [¶](#32-convert-results-into-a-scene-inventory)


In [ ]:
scene_inventory = []

for item in items:
    scene_inventory.append({
        "item_id": item.id,
        "datetime": item.datetime,
        "platform": item.properties.get("platform"),
        "cloud_cover": item.properties.get("eo:cloud_cover"),
        "mgrs_tile": item.properties.get("s2:mgrs_tile"),
    })

scene_inventory = (
    pd.DataFrame(scene_inventory)
    .sort_values("datetime")
    .reset_index(drop=True)
)

scene_inventory

### 3.3 Assign Temporal Periods [¶](#33-assign-temporal-periods)


In [ ]:
scene_inventory["period"] = scene_inventory["datetime"].apply(
    lambda x: (
        "November-February"
        if x.month in [11, 12, 1, 2]
        else "March-April"
        if x.month in [3, 4]
        else "May-June"
        if x.month in [5, 6]
        else None
    )
)

display(
    scene_inventory[
        [
            "datetime",
            "platform",
            "mgrs_tile",
            "cloud_cover",
            "period",
            "item_id",
        ]
    ]
)

### 3.4 Summarize the Scene Inventory [¶](#34-summarize-the-scene-inventory)


In [ ]:
print("\nObservations by period:")
print(scene_inventory["period"].value_counts().sort_index())

print("\nObservations by platform:")
print(scene_inventory["platform"].value_counts())

print("\nCloud-cover statistics:")
print(scene_inventory["cloud_cover"].describe())

### 3.5 Standardize the Sentinel-2 Scene Inventory [¶](#35-normalize-the-sentinel-2-scene-inventory)

The initial STAC query returned multiple Sentinel-2 products for the same acquisition because the GeoSoil-Cuba AOI intersects more than one MGRS tile.

The product identifiers show that the AOI is covered by tiles **17QLF** and **17QMF**.

Before selecting imagery, the catalogue results are therefore standardize by:

* acquisition date;
* MGRS tile;
* Sentinel-2 platform;
* reported cloud cover.

The MGRS tile is extracted from the Sentinel-2 product identifier because the corresponding STAC property is not populated in the returned records.

At this stage, no scene is discarded based solely on cloud percentage. The purpose is to establish the temporal structure of the available observations and determine which acquisitions provide the best coverage of the AOI.


In [ ]:
# Extract MGRS tile from Sentinel-2 product ID
scene_inventory["mgrs_tile"] = scene_inventory["item_id"].str.extract(
    r"_T(\d{2}[A-Z]{3})_"
)[0]

# Convert datetime to date for easier comparison
scene_inventory["acquisition_date"] = (
    pd.to_datetime(scene_inventory["datetime"])
    .dt.date
)

# Reorder columns
scene_inventory = scene_inventory[
    [
        "acquisition_date",
        "datetime",
        "platform",
        "mgrs_tile",
        "cloud_cover",
        "period",
        "item_id",
    ]
]

display(scene_inventory)

### 3.6 Summarize Temporal Coverage by Tile [¶](#36-summarize-temporal-coverage-by-tile)


In [ ]:
tile_summary = (
    scene_inventory
    .groupby(["period", "mgrs_tile"])
    .agg(
        observations=("item_id", "count"),
        first_date=("acquisition_date", "min"),
        last_date=("acquisition_date", "max"),
        mean_cloud=("cloud_cover", "mean"),
        min_cloud=("cloud_cover", "min"),
    )
    .reset_index()
)

display(tile_summary)

### 3.7 Count Distinct Acquisition Dates [¶](#37-count-distinct-acquisition-dates)


In [ ]:
date_summary = (
    scene_inventory
    .groupby(["period", "acquisition_date"])
    .agg(
        tiles=("mgrs_tile", "nunique"),
        platforms=("platform", lambda x: ", ".join(sorted(x.unique()))),
        min_cloud=("cloud_cover", "min"),
        max_cloud=("cloud_cover", "max"),
        mean_cloud=("cloud_cover", "mean"),
    )
    .reset_index()
)

display(date_summary)

In [ ]:
print("Distinct acquisition dates:", len(date_summary))


print("\nNovember–February dates:")
display(
    date_summary[
        date_summary["period"] == "November-February"
    ]
)

print("\nMarch–April dates:")
display(
    date_summary[
        date_summary["period"] == "March-April"
    ]
)

print("\nMay–June dates:")
display(
    date_summary[
        date_summary["period"] == "May-June"
    ]
)

## 4. Initial Acquisition Screening by Cloud Cover [¶](#4-initial-acquisition-screening-by-cloud-cover)

Before downloading Sentinel-2 spectral bands, we perform an initial metadata-based screening of acquisition dates using the mean cloud-cover value across the two MGRS tiles covering the AOI.

This screening is intended to prioritize acquisitions for further processing. It does not determine whether an acquisition is scientifically usable because the scene-level cloud-cover metadata does not describe cloud contamination specifically within the Jaruco AOI.

The screening categories are:

- **Preferred:** mean cloud cover < 20%
- **Candidate:** 20–40%
- **Inspect:** 40–60%
- **Low priority:** > 60%

All categories remain available for later AOI-specific quality assessment using the Sentinel-2 Scene Classification Layer (SCL).


In [ ]:
# Initial metadata-based acquisition screening

date_summary["screening"] = pd.cut(
    date_summary["mean_cloud"],
    bins=[-float("inf"), 20, 40, 60, float("inf")],
    labels=[
        "Preferred",
        "Candidate",
        "Inspect",
        "Low priority"
    ],
    right=False
)

screening_summary = (
    date_summary
    .groupby(["period", "screening"], observed=True)
    .size()
    .reset_index(name="observations")
)

display(screening_summary)

### 4.1 Build the Initial Candidate Acquisition Set [¶](#41-build-the-initial-candidate-acquisition-set)

Based on the metadata screening, we identify acquisitions with mean scene cloud cover below 40% as the initial candidate set for AOI-specific quality assessment.

These acquisitions are not yet considered valid observations. Their suitability will be determined later using the Sentinel-2 Scene Classification Layer (SCL) over the study area.

Acquisitions with 40–60% cloud cover are retained as secondary inspection candidates, while acquisitions above 60% are assigned low priority.


In [ ]:
# Create the initial candidate acquisition set

candidate_dates = (
    date_summary[
        date_summary["mean_cloud"] < 40
    ]
    .copy()
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

inspect_dates = (
    date_summary[
        (date_summary["mean_cloud"] >= 40)
        & (date_summary["mean_cloud"] < 60)
    ]
    .copy()
    .sort_values("acquisition_date")
    .reset_index(drop=True)
)

print("Initial candidate dates (<40% mean cloud):", len(candidate_dates))
print("Secondary inspection dates (40–<60%):", len(inspect_dates))

print("\nCandidate dates by period:")
display(
    candidate_dates.groupby("period").size()
    .reset_index(name="observations")
)

print("\nSecondary inspection dates by period:")
display(
    inspect_dates.groupby("period").size()
    .reset_index(name="observations")
)

### 4.2 Rank Candidate Acquisitions [¶](#42-rank-candidate-acquisitions)

The initial candidate set contains acquisitions with less than 40% mean scene-level cloud cover.

Candidates are ranked by temporal period and mean cloud cover. This ranking is used only to organize the subsequent quality assessment; it does not determine the final set of scientifically valid observations.

The final usable acquisitions will be determined after evaluating cloud and scene-classification conditions within the AOI.


In [ ]:
# Rank candidate acquisitions by period and mean cloud cover

candidate_ranked = (
    candidate_dates[
        [
            "period",
            "acquisition_date",
            "tiles",
            "platforms",
            "min_cloud",
            "max_cloud",
            "mean_cloud",
            "screening",
        ]
    ]
    .sort_values(
        ["period", "mean_cloud", "acquisition_date"]
    )
    .reset_index(drop=True)
)

display(candidate_ranked)

### 4.3 Review Candidate Temporal Coverage [¶](#43-review-candidate-temporal-coverage)

This summary shows the number of initial candidate observations available within each analysis period.

The purpose is to verify that every part of the November 2025–June 2026 study window has sufficient candidate observations before proceeding to raster-level quality assessment.


In [ ]:
# Summarize candidate temporal coverage

candidate_period_summary = (
    candidate_dates
    .groupby("period", observed=True)
    .agg(
        observations=("acquisition_date", "count"),
        first_date=("acquisition_date", "min"),
        last_date=("acquisition_date", "max"),
        best_mean_cloud=("mean_cloud", "min"),
        worst_mean_cloud=("mean_cloud", "max"),
    )
    .reset_index()
)

display(candidate_period_summary)

## 5. AOI-Specific Scene Quality Assessment [¶](#5-aoi-specific-scene-quality-assessment)

The initial acquisition screening was based on scene-level cloud-cover metadata.

We now evaluate acquisition quality within the Jaruco AOI using the Sentinel-2 Scene Classification Layer (SCL).

For each candidate acquisition, the SCL classes associated with clouds, cloud shadows, cirrus, and other unusable pixels will be excluded. The resulting valid-pixel fraction will provide an AOI-specific measure of observation quality.

This assessment is used to determine which acquisition dates provide sufficient cloud-free coverage for subsequent NDVI calculation.

Because the AOI spans MGRS tiles 17QLF and 17QMF, the quality assessment will consider both tiles belonging to each acquisition date.


### 5.1 Retrieve Candidate Sentinel-2 Scene Classification Layers [¶](#51-retrieve-candidate-sentinel-2-scene-classification-layers)

We now retrieve the Scene Classification Layer (SCL) for the candidate Sentinel-2 observations identified during the temporal screening.

Each acquisition date is represented by the two MGRS tiles intersecting the GeoSoil-Cuba AOI: T17QLF and T17QMF.

The SCL is retrieved before downloading B04 and B08 because it provides the pixel-level information needed to evaluate cloud, shadow, water, and other scene conditions over the AOI.

The Copernicus Data Space authentication procedure established during the exploratory analysis is reused here. Credentials are entered interactively and are not stored in the notebook.


In [ ]:
# Authenticate with Copernicus Data Space.
# Credentials are entered interactively and are not stored in the notebook.

import getpass
import requests

username = input("CDSE username: ")
password = getpass.getpass("CDSE password: ")

token_url = (
    "https://identity.dataspace.copernicus.eu/"
    "auth/realms/CDSE/protocol/openid-connect/token"
)

token_response = requests.post(
    token_url,
    data={
        "client_id": "cdse-public",
        "grant_type": "password",
        "username": username,
        "password": password,
    },
    timeout=30,
)

print("Status code:", token_response.status_code)

token_response.raise_for_status()

token_data = token_response.json()
access_token = token_data["access_token"]

headers = {
    "Authorization": f"Bearer {access_token}"
}

print("Access token obtained successfully.")
print(
    "Refresh token available:",
    "yes" if "refresh_token" in token_data else "no"
)

### 5.2 Retrieve Candidate Sentinel-2 Observations [¶](#52-retrieve-candidate-sentinel-2-observations)

Use the existing candidate_dates/date_summary from the screening section. This searches CDSE for the two tiles for every candidate date.


In [ ]:
# Inspect the columns and first rows of date_summary.

print("Columns:")
print(date_summary.columns.tolist())

display(date_summary.head())

In [ ]:
# Retrieve Sentinel-2 observations for the candidate dates.

from pystac_client import Client

catalog = Client.open(
    "https://stac.dataspace.copernicus.eu/v1"
)

tiles = ["17QLF", "17QMF"]

date_summary["acquisition_date"] = pd.to_datetime(
    date_summary["acquisition_date"]
)

candidate_dates = (
    date_summary.loc[
        date_summary["screening"].isin(["Preferred", "Candidate"]),
        "acquisition_date"
    ]
    .dt.strftime("%Y-%m-%d")
    .tolist()
)

print(f"Candidate dates: {len(candidate_dates)}")
print(f"Tiles: {tiles}")

### 5.3 Search CDSE for Candidate Observations [¶](#53-search-cdse-for-candidate-observations)


In [ ]:
# Build the list of candidate acquisition dates.

candidate_dates = (
    date_summary.loc[
        date_summary["screening"].isin(["Preferred", "Candidate"]),
        "acquisition_date"
    ]
    .dt.strftime("%Y-%m-%d")
    .tolist()
)

print(f"Candidate dates: {len(candidate_dates)}")
print(candidate_dates[:10])

In [ ]:
# Search the CDSE STAC catalogue once for the complete study period.
#
# This retrieves metadata only.
# No Sentinel-2 raster files are downloaded here.

SEARCH_START = "2025-11-01T00:00:00Z"
SEARCH_END = "2026-06-30T23:59:59Z"

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=AOI_BBOX_WGS84,
    datetime=f"{SEARCH_START}/{SEARCH_END}",
    max_items=500,
)

items = list(search.items())

print(f"Sentinel-2 L2A observations found: {len(items)}")

In [ ]:
# Inspect the properties returned by the CDSE STAC search.

item = items[0]

print("Item ID:")
print(item.id)

print("\nProperties:")
for key, value in item.properties.items():
    print(f"{key}: {value}")

In [ ]:
# Inspect the first item's complete STAC properties dictionary.

item.properties

In [ ]:
# Filter observations to the two Sentinel-2 tiles covering the AOI

target_tiles = {"17QLF", "17QMF"}

target_items = []

for item in items:
    tile = item.properties.get("grid:code", "").replace("MGRS-", "")

    if tile in target_tiles:
        target_items.append(item)

print(f"Observations for target tiles: {len(target_items)}")

In [ ]:
# Keep only observations corresponding to the initially screened candidate dates

candidate_items = []

for item in target_items:
    acquisition_date = item.datetime.strftime("%Y-%m-%d")

    if acquisition_date in candidate_dates:
        candidate_items.append({
            "acquisition_date": acquisition_date,
            "tile": item.properties.get("grid:code", "").replace("MGRS-", ""),
            "item_id": item.id,
            "item": item
        })

print(f"Candidate observations: {len(candidate_items)}")

In [ ]:
# Inspect the candidate acquisition inventory

candidate_inventory = pd.DataFrame([
    {
        "acquisition_date": record["acquisition_date"],
        "tile": record["tile"],
        "item_id": record["item_id"],
        "cloud_cover": record["item"].properties.get("eo:cloud_cover"),
    }
    for record in candidate_items
])

candidate_inventory = (
    candidate_inventory
    .sort_values(["acquisition_date", "tile"])
    .reset_index(drop=True)
)

print(f"Candidate observations: {len(candidate_inventory)}")
print(
    f"Unique candidate dates: "
    f"{candidate_inventory['acquisition_date'].nunique()}"
)

print("\nObservations per date:")
print(
    candidate_inventory
    .groupby("acquisition_date")
    .size()
    .value_counts()
    .sort_index()
)

candidate_inventory

### 5.4 Standardize Candidate Acquisitions by Date and Satellite [¶](#54-normalize-candidate-acquisitions-by-date-and-satellite)

The initial screening identified **63 candidate acquisition dates** across the two Sentinel-2 tiles covering the AOI: **T17QLF** and **T17QMF**.

Most dates contain one Sentinel-2 acquisition, producing two tile observations per date. However, five dates — **2025-11-25, 2026-01-04, 2026-03-15, 2026-04-04, and 2026-06-23** — contain observations from both **Sentinel-2A and Sentinel-2B**. These dates therefore produce four tile observations rather than two.

This results in **136 candidate tile observations**, representing **68 date–satellite acquisition pairs**.

For temporal NDVI analysis, the acquisition pair is treated as the basic unit for these dual-satellite dates. The two tiles should preferably be selected from the **same satellite acquisition**, rather than independently selecting the lowest-cloud observation for each tile. This maintains temporal and observational consistency between T17QLF and T17QMF before they are mosaicked over the AOI.

At this stage, both S2A and S2B alternatives are retained. Cloud-cover metadata is used only for preliminary screening; the final selection will be based on **AOI-specific Scene Classification Layer (SCL) quality**.

The resulting structure is therefore:

**acquisition date → satellite → T17QLF + T17QMF → AOI mosaic**

This normalization reduces the 136 tile observations to **68 candidate date–satellite acquisition pairs** while preserving all potentially useful alternatives for the subsequent SCL quality assessment.


In [ ]:
# Identify dates with more than one observation per tile

duplicate_dates = (
    candidate_inventory
    .groupby(["acquisition_date", "tile"])
    .size()
    .reset_index(name="count")
)

duplicate_dates = duplicate_dates[
    duplicate_dates["count"] > 1
]

duplicate_dates

In [ ]:
# Show the product records for dates with duplicate observations

duplicate_date_list = duplicate_dates["acquisition_date"].unique()

candidate_inventory[
    candidate_inventory["acquisition_date"].isin(duplicate_date_list)
].sort_values(["acquisition_date", "tile", "item_id"])

In [ ]:
# Build one row per acquisition date and satellite

acquisition_pairs = (
    candidate_inventory
    .assign(
        satellite=lambda df: df["item_id"].str.extract(r"^(S2[A-C])_")[0]
    )
    .groupby(["acquisition_date", "satellite"], as_index=False)
    .agg(
        mean_tile_cloud=("cloud_cover", "mean"),
        max_tile_cloud=("cloud_cover", "max"),
        tile_count=("tile", "nunique"),
    )
    .sort_values(["acquisition_date", "mean_tile_cloud"])
    .reset_index(drop=True)
)

acquisition_pairs

In [ ]:
print(f"Acquisition pairs: {len(acquisition_pairs)}")
print(
    f"Dates represented: "
    f"{acquisition_pairs['acquisition_date'].nunique()}"
)

print("\nAcquisitions with both tiles:")
print(
    acquisition_pairs["tile_count"]
    .value_counts()
    .sort_index()
)

## 6. Prepare and Download Sentinel-2 SCL Assets [¶](#6-prepare-and-download-sentinel-2-scl-assets)

The candidate inventory contains **68 date–satellite acquisition pairs**, each with complete coverage from the two Sentinel-2 MGRS tiles intersecting the Jaruco AOI: **T17QLF** and **T17QMF**.

For the quality-assessment stage, only the **Scene Classification Layer (SCL)** is required. The SCL provides pixel-level classification information that can be used to identify clouds, cloud shadows, cirrus, saturated or defective pixels, and other surface classes.

The SCL is provided at **20 m spatial resolution**, whereas Sentinel-2 red (B04) and near-infrared (B08) bands used later for NDVI are available at 10 m. The SCL will therefore be used here for acquisition screening rather than for calculating NDVI itself.

For each acquisition pair, the SCL asset will be retrieved for both T17QLF and T17QMF. The resulting SCL rasters will then be evaluated after intersection with the study AOI.

At this stage, **no acquisition is permanently excluded based solely on metadata cloud cover**. The purpose of the SCL assessment is to determine the actual proportion and spatial distribution of usable pixels within the AOI.


In [ ]:
# Inspect the asset keys available in a Sentinel-2 L2A STAC item

item = candidate_items[0]["item"]

print("Item ID:")
print(item.id)

print("\nAvailable assets:")
for key, asset in item.assets.items():
    print(f"{key}: {asset.title}")

In [ ]:
# Show the asset keys only

print(list(item.assets.keys()))

### 6.1 Build the Sentinel-2 SCL Inventory [¶](#61-build-the-sentinel-2-scl-inventory)

The current Copernicus Data Space STAC records expose the Scene Classification Layer using the asset key **`SCL_20m`**. Both 20 m and 60 m SCL products are available, but the **20 m product** is used for this analysis because it provides finer spatial detail for evaluating cloud, cloud-shadow, cirrus, and valid-surface coverage within the AOI.

The SCL inventory is constructed from the already screened candidate observations. Each record represents one Sentinel-2 tile acquisition and contains the acquisition date, satellite, MGRS tile, product identifier, scene-level cloud-cover metadata, and the SCL asset URL.

The expected result is **136 SCL tile observations representing 68 complete date–satellite acquisition pairs**.


In [ ]:
# Build the SCL inventory using the current CDSE asset name: SCL_20m

scl_inventory = []

for record in candidate_items:

    item = record["item"]

    acquisition_date = record["acquisition_date"]
    tile = record["tile"]

    satellite = (
        item.properties
        .get("platform", "")
        .replace("sentinel-", "")
        .upper()
    )

    # Current CDSE STAC asset name
    scl_asset = item.assets.get("SCL_20m")

    if scl_asset is None:
        print(f"SCL_20m asset not found: {record['item_id']}")
        continue

    scl_inventory.append({
        "acquisition_date": acquisition_date,
        "satellite": satellite,
        "tile": tile,
        "item_id": record["item_id"],
        "cloud_cover": item.properties.get("eo:cloud_cover"),
        "scl_href": scl_asset.href,
    })


scl_inventory = (
    pd.DataFrame(scl_inventory)
    .sort_values(
        ["acquisition_date", "satellite", "tile"]
    )
    .reset_index(drop=True)
)

print(f"SCL observations: {len(scl_inventory)}")

print(
    "Acquisition pairs represented:",
    scl_inventory[
        ["acquisition_date", "satellite"]
    ].drop_duplicates().shape[0]
)

print("\nObservations per acquisition pair:")
print(
    scl_inventory
    .groupby(["acquisition_date", "satellite"])
    .size()
    .value_counts()
    .sort_index()
)

### 6.2 Download Sentinel-2 SCL 20 m Rasters [¶](#62-download-sentinel-2-scl-20-m-rasters)

The validated SCL inventory contains **136 tile observations representing 68 date–satellite acquisition pairs**. Each acquisition pair contains one SCL raster for T17QLF and one for T17QMF.

Only the **20 m Scene Classification Layer (SCL_20m)** is downloaded at this stage. The red and near-infrared bands required for NDVI are intentionally deferred until the SCL quality assessment identifies the acquisitions suitable for analysis.

The download uses the authenticated Copernicus Data Space HTTPS endpoint established earlier. Authentication is performed once per notebook session, and the resulting access token is used for the individual raster downloads.

Downloaded files are stored in the satellite-data directory and are named using the acquisition date, Sentinel-2 tile, and SCL product designation.


In [ ]:
# Inspect one SCL_20m asset before downloading

test_record = scl_inventory.iloc[0]

print("Acquisition date:", test_record["acquisition_date"])
print("Satellite:", test_record["satellite"])
print("Tile:", test_record["tile"])
print("Item ID:", test_record["item_id"])
print("\nSCL URL:")
print(test_record["scl_href"])

In [ ]:
# Get the authenticated HTTPS endpoint for one SCL_20m asset

test_record = scl_inventory.iloc[0]

item = next(
    record["item"]
    for record in candidate_items
    if record["item_id"] == test_record["item_id"]
)

scl_asset = item.assets["SCL_20m"]

https_url = scl_asset.extra_fields["alternate"]["https"]["href"]

print("SCL HTTPS endpoint:")
print(https_url)

In [ ]:
# Test authenticated access to the SCL_20m raster asset

headers = {
    "Authorization": f"Bearer {access_token}"
}

response = requests.get(
    https_url,
    headers=headers,
    stream=True,
    allow_redirects=True,
    timeout=30
)

print("Status code:", response.status_code)
print("Final URL:", response.url)
print("Content type:", response.headers.get("Content-Type"))
print("Content length:", response.headers.get("Content-Length"))

### 6.3 Test a Single Sentinel-2 SCL Download [¶](#63-test-a-single-sentinel-2-scl-download)

We first download one authenticated Scene Classification Layer (SCL) raster as a local test. This confirms that the complete path from CDSE STAC → authenticated HTTPS endpoint → local JP2 file works before downloading the full candidate set.


In [ ]:
from pathlib import Path
import requests

satellite_dir = RAW_SATELLITE_DIR

satellite_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
## Download one Sentinel-2 SCL raster

test_date = test_record["acquisition_date"]
test_satellite = test_record["satellite"]
test_tile = test_record["tile"]

scl_path = (
    satellite_dir
    / f"{test_date}_{test_satellite}_{test_tile}_SCL_20m.jp2"
)

with requests.get(
    https_url,
    headers=headers,
    stream=True,
    allow_redirects=True,
    timeout=60
) as response:

    response.raise_for_status()

    with open(scl_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Downloaded:")
print(scl_path)
print(f"File size: {scl_path.stat().st_size:,} bytes")


## Validate the downloaded SCL raster


import rasterio

with rasterio.open(scl_path) as src:

    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Dtype:", src.dtypes[0])
    print("Count:", src.count)
    print("NoData:", src.nodata)
    print("Bounds:", src.bounds)


## Inspect SCL class values

import numpy as np

with rasterio.open(scl_path) as src:
    scl = src.read(1)

values, counts = np.unique(scl, return_counts=True)

print("SCL classes found:")
for value, count in zip(values, counts):
    print(f"{value:>2}: {count:,} pixels")



### 6.4 Download Candidate SCL Rasters [¶](#64-download-candidate-scl-rasters)

The candidate inventory contains 136 SCL observations representing 68 acquisition pairs across 63 dates. We download each SCL raster using the authenticated HTTPS endpoint supplied by the CDSE STAC asset.


In [ ]:

from pathlib import Path
import requests
import time

satellite_dir = RAW_SATELLITE_DIR

satellite_dir.mkdir(parents=True, exist_ok=True)

downloaded = 0
skipped = 0
failed = []

for i, record in enumerate(scl_inventory.itertuples(index=False), start=1):

    acquisition_date = record.acquisition_date
    satellite = record.satellite
    tile = record.tile
    item_id = record.item_id

    # Find the corresponding STAC item
    item = next(
        candidate["item"]
        for candidate in candidate_items
        if candidate["item_id"] == item_id
    )

    # Get the 20 m SCL asset
    scl_asset = item.assets["SCL_20m"]

    # Use the authenticated HTTPS endpoint supplied by STAC
    https_url = scl_asset.extra_fields["alternate"]["https"]["href"]

    # Unique filename: date + satellite + tile
    scl_path = (
        satellite_dir
        / f"{acquisition_date}_{satellite}_{tile}_SCL_20m.jp2"
    )

    # Do not download a file that already exists
    if scl_path.exists() and scl_path.stat().st_size > 0:
        skipped += 1
        print(
            f"[{i:03d}/136] SKIP  "
            f"{acquisition_date} {satellite} {tile} "
            f"(already exists)"
        )
        continue

    try:
        with requests.get(
            https_url,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=120
        ) as response:

            response.raise_for_status()

            with open(scl_path, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

        downloaded += 1

        print(
            f"[{i:03d}/136] OK    "
            f"{acquisition_date} {satellite} {tile} "
            f"({scl_path.stat().st_size:,} bytes)"
        )

    except Exception as e:

        failed.append({
            "acquisition_date": acquisition_date,
            "satellite": satellite,
            "tile": tile,
            "item_id": item_id,
            "error": str(e),
        })

        print(
            f"[{i:03d}/136] FAILED "
            f"{acquisition_date} {satellite} {tile}: {e}"
        )

    # Small pause between requests
    time.sleep(0.2)


print("\nDownload summary")
print("----------------")
print(f"Downloaded: {downloaded}")
print(f"Skipped:    {skipped}")
print(f"Failed:     {len(failed)}")
print(f"Total:      {downloaded + skipped + len(failed)}")



### 6.5 Check the Download Directory [¶](#65-check-the-download-directory)

After the download completes, verify that the expected number of SCL files exists.


In [ ]:
scl_files = sorted(
    satellite_dir.glob("*_SCL_20m.jp2")
)

print(f"SCL files found: {len(scl_files)}")

### 6.6 Inspect Failed Downloads [¶](#66-inspect-failed-downloads)

If anything failed, this displays the failures without stopping the whole workflow.


In [ ]:

if failed:
    print("\nFailed downloads:")
    
    for failure in failed:
        print(
            f"{failure['acquisition_date']} "
            f"{failure['satellite']} "
            f"{failure['tile']}: "
            f"{failure['error']}"
        )
else:
    print("No download failures.")


### 6.7 Confirm the Expected Download Count [¶](#67-confirm-the-expected-download-count)


In [ ]:

expected = len(scl_inventory)

print(f"Expected observations: {expected}")
print(f"Downloaded files:      {len(scl_files)}")

if len(scl_files) == expected:
    print("✓ All candidate SCL observations are present.")
else:
    print("⚠ File count does not match the candidate inventory.")


## 7. Validate Downloaded Sentinel-2 SCL Rasters [¶](#7-validate-downloaded-sentinel-2-scl-rasters)

We validate every downloaded SCL raster before using it for AOI-specific quality assessment. The validation checks file readability, CRS, spatial resolution, band count, dimensions, data type, and SCL class values.


In [ ]:
from pathlib import Path
import rasterio
import numpy as np
import pandas as pd

scl_files = sorted(
    satellite_dir.glob("*_SCL_20m.jp2")
)

print(f"SCL files found: {len(scl_files)}")

validation_results = []

for scl_path in scl_files:

    result = {
        "file": scl_path.name,
        "path": str(scl_path),
        "readable": False,
        "crs": None,
        "width": None,
        "height": None,
        "res_x": None,
        "res_y": None,
        "count": None,
        "dtype": None,
        "nodata": None,
        "min": None,
        "max": None,
        "classes": None,
        "error": None,
    }

    try:
        with rasterio.open(scl_path) as src:

            result["readable"] = True
            result["crs"] = src.crs.to_string() if src.crs else None
            result["width"] = src.width
            result["height"] = src.height
            result["res_x"] = src.res[0]
            result["res_y"] = src.res[1]
            result["count"] = src.count
            result["dtype"] = src.dtypes[0]
            result["nodata"] = src.nodata

            # Read the complete SCL raster
            scl = src.read(1)

            valid = scl[scl != src.nodata] if src.nodata is not None else scl

            if valid.size > 0:
                result["min"] = int(valid.min())
                result["max"] = int(valid.max())

                classes = np.unique(valid)
                result["classes"] = ",".join(
                    str(int(value)) for value in classes
                )

    except Exception as e:
        result["error"] = str(e)

    validation_results.append(result)


scl_validation = pd.DataFrame(validation_results)

print(f"Validated: {len(scl_validation)}")


### 7.1 Inspect the Validation Summary [¶](#71-inspect-the-validation-summary)


In [ ]:

scl_validation[
    [
        "file",
        "readable",
        "crs",
        "width",
        "height",
        "res_x",
        "res_y",
        "count",
        "dtype",
        "min",
        "max",
        "classes",
        "error",
    ]
].head()


### 7.2 Check Structural Consistency [¶](#72-check-structural-consistency)


In [ ]:
print("Readable:")
print(scl_validation["readable"].value_counts())

print("\nCRS:")
print(scl_validation["crs"].value_counts())

print("\nDimensions:")
print(
    scl_validation[
        ["width", "height"]
    ].value_counts()
)

print("\nResolution:")
print(
    scl_validation[
        ["res_x", "res_y"]
    ].value_counts()
)

print("\nBand count:")
print(scl_validation["count"].value_counts())

print("\nData type:")
print(scl_validation["dtype"].value_counts())


### 7.3 Check for Unexpected SCL Values [¶](#73-check-for-unexpected-scl-values)

The Sentinel-2 SCL class values are integer codes. We flag any values outside the expected range rather than silently assuming that every raster is correct.


In [ ]:
expected_scl_classes = set(range(12))

unexpected = []

for _, row in scl_validation.iterrows():

    if not row["readable"] or row["classes"] is None:
        continue

    classes = {
        int(value)
        for value in row["classes"].split(",")
    }

    unexpected_classes = classes - expected_scl_classes

    if unexpected_classes:
        unexpected.append({
            "file": row["file"],
            "unexpected_classes": sorted(unexpected_classes),
        })

if unexpected:
    print("Unexpected SCL values found:")
    for item in unexpected:
        print(item)
else:
    print("No unexpected SCL class values found.")




### 7.4 Identify Failed or Inconsistent Files [¶](#74-identify-failed-or-inconsistent-files)


In [ ]:
problems = scl_validation[
    (~scl_validation["readable"])
    | (scl_validation["crs"] != "EPSG:32617")
    | (scl_validation["res_x"] != 20.0)
    | (scl_validation["res_y"] != 20.0)
    | (scl_validation["count"] != 1)
    | (scl_validation["dtype"] != "uint8")
]

print(f"Files requiring inspection: {len(problems)}")

if len(problems) > 0:
    display(problems)
else:
    print("✓ All SCL rasters pass the structural validation.")

## 8. Perform AOI-Specific SCL Screening [¶](#8-perform-aoi-specific-scl-screening)

Scene-level cloud cover from the Sentinel-2 metadata describes the entire satellite scene. For this project, we instead evaluate the SCL classes over the Jaruco AOI.

The two MGRS tiles are combined for each acquisition so that the assessment represents the complete AOI.


In [ ]:
import fiona 

# Curated AOI layer
aoi_path = (
    PROJECT_DIR / "data" / "processed" / "vector" / "soil_vegetation_analysis.gpkg"
)


print(fiona.listlayers(aoi_path))

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd
from pathlib import Path


aoi = gpd.read_file(
    aoi_path,
    layer="soil_vegetation"
)

print("AOI CRS:", aoi.crs)
print("AOI features:", len(aoi))



### 8.1 Prepare the AOI Geometry [¶](#81-prepare-the-aoi-geometry)

The SCL rasters use EPSG:32617, so the AOI is reprojected to the raster CRS before masking.


In [ ]:
with rasterio.open(scl_files[0]) as src:
    scl_crs = src.crs

aoi_scl = aoi.to_crs(scl_crs)

aoi_geometry = [
    geom.__geo_interface__
    for geom in aoi_scl.geometry
]

print("AOI CRS for SCL:", aoi_scl.crs)


### 8.2 Define SCL Classes for Quality Assessment [¶](#82-define-scl-classes-for-quality-assessment)

We only need a small number of categories for this portfolio analysis.


In [ ]:
SCL_CLASSES = {
    3: "Cloud shadow",
    4: "Vegetation",
    5: "Not vegetated",
    6: "Water",
    7: "Unclassified",
    8: "Medium-probability cloud",
    9: "High-probability cloud",
    10: "Cirrus",
    11: "Snow/ice",
}

CLOUD_CLASSES = {8, 9, 10}
SHADOW_CLASSES = {3}

# Pixels considered usable for the NDVI workflow.
# Vegetation and bare/non-vegetated surfaces are both retained.
USABLE_CLASSES = {4, 5}


### 8.3 Calculate AOI-Specific SCL Statistics [¶](#83-calculate-aoi-specific-scl-statistics)

Each acquisition pair contains the two tiles needed to cover the AOI.


In [ ]:
import geopandas as gpd 
import rasterio 
from rasterio.merge import merge 
from rasterio.mask import mask 
import numpy as np 
import pandas as pd 
from pathlib import Path

results = []

for (acquisition_date, satellite), group in scl_inventory.groupby(
    ["acquisition_date", "satellite"]
):

    tile_paths = []

    for _, record in group.iterrows():

        scl_path = (
            satellite_dir
            / f"{record['acquisition_date']}_"
              f"{record['satellite']}_"
              f"{record['tile']}_SCL_20m.jp2"
        )

        tile_paths.append(scl_path)

    # Open both SCL tiles
    sources = [
        rasterio.open(path)
        for path in tile_paths
    ]

    try:

        # Mosaic the two tiles
        mosaic, mosaic_transform = merge(
            sources,
            nodata=255,
            method="first"
        )

        mosaic_array = mosaic[0]

        # Build a temporary raster profile
        profile = sources[0].profile.copy()

        profile.update(
            height=mosaic_array.shape[0],
            width=mosaic_array.shape[1],
            transform=mosaic_transform,
            nodata=255
        )

        # Mask the mosaic to the AOI
        with rasterio.io.MemoryFile() as memfile:

            with memfile.open(**profile) as temp:

                temp.write(mosaic_array, 1)

                scl_aoi, _ = mask(
                    temp,
                    aoi_geometry,
                    crop=True,
                    filled=True,
                    nodata=255
                )

        values = scl_aoi[0]

        # Remove pixels outside the AOI
        values = values[values != 255]

        total = len(values)

        if total == 0:
            continue

        def pct(classes):
            return (
                100
                * np.isin(values, list(classes)).sum()
                / total
            )

        results.append({
            "acquisition_date": acquisition_date,
            "satellite": satellite,
            "aoi_pixels": total,
            "usable_pct": pct(USABLE_CLASSES),
            "vegetation_pct": pct({4}),
            "water_pct": pct({6}),
            "cloud_shadow_pct": pct(SHADOW_CLASSES),
            "cloud_pct": pct(CLOUD_CLASSES),
            "unclassified_pct": pct({7}),
            "saturated_pct": pct({1}),
        })

    finally:

        for src in sources:
            src.close()


aoi_scl_summary = pd.DataFrame(results)

print(
    "Acquisition pairs evaluated:",
    len(aoi_scl_summary)
)

aoi_scl_summary.head()

### 8.4 Inspect AOI-Specific Results [¶](#84-inspect-aoi-specific-results)


In [ ]:
aoi_scl_summary.sort_values(
    ["acquisition_date", "satellite"]
).head(200)

### 8.5 Classify Acquisition Quality [¶](#85-classify-acquisition-quality)

We can now classify acquisitions based on AOI cloud percentage, rather than the original scene-wide cloud percentage.


In [ ]:
aoi_scl_summary["screening"] = pd.cut(
    aoi_scl_summary["cloud_pct"],
    bins=[-float("inf"), 10, 25, 40, float("inf")],
    labels=[
        "Preferred",
        "Candidate",
        "Inspect",
        "Low priority"
    ],
    right=False
)

### 8.6 Summarize the Screening Results [¶](#86-summarize-the-screening-results)


In [ ]:
print(
    aoi_scl_summary["screening"]
    .value_counts()
    .sort_index()
)

### 8.7 Review the Best AOI Conditions [¶](#87-review-the-best-aoi-conditions)


In [ ]:
pd.set_option("display.max_rows", None)

aoi_scl_summary.sort_values(
    ["cloud_pct", "cloud_shadow_pct"]
)[
    [
        "acquisition_date",
        "satellite",
        "usable_pct",
        "vegetation_pct",
        "water_pct",
        "cloud_pct",
        "cloud_shadow_pct",
        "unclassified_pct",
        "screening",
    ]
].head(70)

## 9. Select Representative NDVI Acquisitions [¶](#9-select-representative-ndvi-acquisitions)


In [ ]:
# Keep only Preferred and Candidate acquisitions
ndvi_candidates = aoi_scl_summary[
    aoi_scl_summary["cloud_pct"] < 30
].copy()

# Convert dates to datetime
ndvi_candidates["acquisition_date"] = pd.to_datetime(
    ndvi_candidates["acquisition_date"]
)

# If both Sentinel-2 satellites are available on the same date,
# keep the satellite with the lower AOI cloud percentage.
ndvi_candidates = (
    ndvi_candidates
    .sort_values(
        ["acquisition_date", "cloud_pct"]
    )
    .drop_duplicates(
        subset=["acquisition_date"],
        keep="first"
    )
)

print(
    "Candidate acquisition dates:",
    len(ndvi_candidates)
)

### 9.1 Select Temporally Spaced Dates [¶](#91-select-temporally-spaced-dates)


In [ ]:
periods = {
    "November-February": {
        "start": "2025-11-01",
        "end": "2026-02-28",
        "n": 4,
    },
    "March-April": {
        "start": "2026-03-01",
        "end": "2026-04-30",
        "n": 14,
    },
    "May-June": {
        "start": "2026-05-01",
        "end": "2026-06-30",
        "n": 8,
    },
}

selected_parts = []

for period, settings in periods.items():

    period_data = ndvi_candidates[
        (ndvi_candidates["acquisition_date"] >= settings["start"])
        & (ndvi_candidates["acquisition_date"] <= settings["end"])
    ].copy()

    period_data = period_data.sort_values("acquisition_date")

    # Divide the available dates into approximately equal temporal
    # sections and select the clearest acquisition from each section.
    bins = np.linspace(
        0,
        len(period_data),
        settings["n"] + 1
    ).astype(int)

    selected_indices = []

    for start, end in zip(bins[:-1], bins[1:]):

        section = period_data.iloc[start:end]

        if len(section) == 0:
            continue

        best = section.loc[
            section["cloud_pct"].idxmin()
        ]

        selected_indices.append(best.name)

    selected_period = period_data.loc[
        selected_indices
    ].copy()

    selected_period["period"] = period

    selected_parts.append(selected_period)


ndvi_selection = pd.concat(
    selected_parts,
    ignore_index=True
)

ndvi_selection = ndvi_selection.sort_values(
    "acquisition_date"
).reset_index(drop=True)

print(
    "Selected acquisition pairs:",
    len(ndvi_selection)
)

### 9.2 Interpret Cloud Screening as a Ranking Aid [¶](#92-interpret-cloud-screening-as-a-ranking-aid)


In [ ]:
# Define the three project periods and the number of
# acquisitions to select from each period.

periods = {
    "November-February": {
        "start": "2025-11-01",
        "end": "2026-02-28",
        "n": 4,
    },
    "March-April": {
        "start": "2026-03-01",
        "end": "2026-04-30",
        "n": 16,
    },
    "May-June": {
        "start": "2026-05-01",
        "end": "2026-06-30",
        "n": 8,
    },
}

selected_parts = []

for period, settings in periods.items():

    period_data = ndvi_candidates[
        (ndvi_candidates["acquisition_date"] >= settings["start"])
        & (ndvi_candidates["acquisition_date"] <= settings["end"])
    ].copy()

    period_data = period_data.sort_values("acquisition_date")

    # Divide the available dates into approximately equal
    # temporal sections.
    bins = np.linspace(
        0,
        len(period_data),
        settings["n"] + 1
    ).astype(int)

    selected_indices = []

    for start, end in zip(bins[:-1], bins[1:]):

        section = period_data.iloc[start:end]

        if len(section) == 0:
            continue

        # Select the acquisition closest to the middle
        # of the temporal section.
        middle = len(section) // 2

        selected = section.iloc[middle]

        selected_indices.append(selected.name)

    selected_period = period_data.loc[
        selected_indices
    ].copy()

    selected_period["period"] = period

    selected_parts.append(selected_period)


ndvi_selection = pd.concat(
    selected_parts,
    ignore_index=True
)

ndvi_selection = ndvi_selection.sort_values(
    "acquisition_date"
).reset_index(drop=True)

print(
    "Selected acquisition pairs:",
    len(ndvi_selection)
)

### 9.3 Inspect Selected Acquisitions [¶](#93-inspect-selected-acquisitions)


In [ ]:
ndvi_selection[
    [
        "acquisition_date",
        "satellite",
        "cloud_pct",
        "cloud_shadow_pct",
        "usable_pct",
        "vegetation_pct",
        "water_pct",
        "screening",
        "period",
    ]
]

### 9.4 Verify Temporal Distribution [¶](#94-verify-temporal-distribution)


In [ ]:
print(
    ndvi_selection["period"]
    .value_counts()
    .reindex(periods.keys())
)

### 9.5 Create the Final Acquisition List [¶](#95-create-the-final-acquisition-list)


In [ ]:
selected_acquisitions = (
    ndvi_selection[
        [
            "acquisition_date",
            "satellite",
            "period",
        ]
    ]
    .to_dict("records")
)

print("Selected acquisitions:")

for acquisition in selected_acquisitions:
    print(
        acquisition["acquisition_date"].strftime("%Y-%m-%d"),
        acquisition["satellite"],
        acquisition["period"]
    )

## 10. Download and Inventory Sentinel-2 B04 and B08 Bands [¶](#10-download-and-inventory-sentinel-2-b04-and-b08-bands)

This cell downloads B04 (red) and B08 (NIR) for the 15 selected acquisition pairs.

The satellite names in `ndvi_selection` (`2A`, `2B`, `2C`) are converted to the corresponding CDSE STAC platform names (`sentinel-2a`, `sentinel-2b`, `sentinel-2c`).

Completed downloads are skipped. Interrupted downloads use temporary `.part` files and are safely restarted on the next run.


In [ ]:
# Inspect the selected acquisition table

print(ndvi_selection.columns.tolist())
print()
print(ndvi_selection.head())


In [ ]:
# Inspect satellite/platform naming in candidate_items

for record in candidate_items[:10]:
    item = record["item"]

    print(
        record["acquisition_date"],
        record["tile"],
        "| satellite:",
        item.properties.get("platform"),
        "| constellation:",
        item.properties.get("constellation"),
        "| item_id:",
        record["item_id"]
    )

In [ ]:

import pandas as pd
import requests
from pathlib import Path

# Directory for Sentinel-2 spectral bands
satellite_dir = RAW_SATELLITE_DIR

satellite_dir.mkdir(parents=True, exist_ok=True)

headers = {
    "Authorization": f"Bearer {access_token}"
}

bands = ["B04_10m", "B08_10m"]

# Convert the short satellite names used in ndvi_selection
# to the platform names used by the CDSE STAC records.
satellite_mapping = {
    "2A": "sentinel-2a",
    "2B": "sentinel-2b",
    "2C": "sentinel-2c"
}

download_records = []

for _, selection in ndvi_selection.iterrows():

    acquisition_date = pd.to_datetime(
        selection["acquisition_date"]
    ).strftime("%Y-%m-%d")

    satellite = selection["satellite"]

    platform = satellite_mapping[satellite]

    # Find the two tile observations belonging to this
    # acquisition date and satellite.
    pair_items = [
        record
        for record in candidate_items
        if pd.to_datetime(
            record["acquisition_date"]
        ).strftime("%Y-%m-%d") == acquisition_date
        and record["item"].properties.get("platform") == platform
    ]

    if len(pair_items) == 0:

        print(
            f"No matching items found: "
            f"{acquisition_date} {satellite}"
        )

        continue

    for record in pair_items:

        item = record["item"]
        tile = record["tile"]

        for band in bands:

            asset = item.assets.get(band)

            if asset is None:

                print(
                    f"Missing {band}: "
                    f"{acquisition_date} "
                    f"{satellite} "
                    f"{tile}"
                )

                continue

            # CDSE authenticated HTTPS endpoint
            https_url = (
                asset.extra_fields["alternate"]["https"]["href"]
            )

            output_path = (
                satellite_dir
                / f"{acquisition_date}_"
                  f"{satellite}_"
                  f"{tile}_"
                  f"{band}.jp2"
            )

            # Temporary file used during download
            temp_path = output_path.with_suffix(".part")

            # Skip completed downloads
            if output_path.exists():

                print(
                    f"Already exists: "
                    f"{output_path.name}"
                )

                download_records.append(
                    {
                        "acquisition_date": acquisition_date,
                        "satellite": satellite,
                        "tile": tile,
                        "band": band,
                        "path": str(output_path),
                        "size_bytes": output_path.stat().st_size,
                        "status": "already_exists"
                    }
                )

                continue

            # Remove an incomplete download from a previous run
            if temp_path.exists():

                print(
                    f"Removing incomplete file: "
                    f"{temp_path.name}"
                )

                temp_path.unlink()

            print(
                f"Downloading: "
                f"{output_path.name}"
            )

            try:

                with requests.get(
                    https_url,
                    headers=headers,
                    stream=True,
                    allow_redirects=True,
                    timeout=60
                ) as response:

                    response.raise_for_status()

                    with open(temp_path, "wb") as f:

                        for chunk in response.iter_content(
                            chunk_size=1024 * 1024
                        ):
                            if chunk:
                                f.write(chunk)

                # Rename only after successful completion
                temp_path.rename(output_path)

                size_bytes = output_path.stat().st_size

                download_records.append(
                    {
                        "acquisition_date": acquisition_date,
                        "satellite": satellite,
                        "tile": tile,
                        "band": band,
                        "path": str(output_path),
                        "size_bytes": size_bytes,
                        "status": "downloaded"
                    }
                )

                print(
                    f"  Saved: "
                    f"{size_bytes / (1024**2):.1f} MB"
                )

            except Exception as e:

                # Remove incomplete file
                if temp_path.exists():
                    temp_path.unlink()

                print(
                    f"  ERROR: "
                    f"{output_path.name}"
                )

                print(f"  {e}")

print("\nDownload stage complete.")

### 10.1 Inventory Downloaded B04 and B08 Rasters [¶](#101-inventory-downloaded-b04-and-b08-rasters)


In [ ]:
from pathlib import Path
import pandas as pd

satellite_dir = RAW_SATELLITE_DIR

band_files = sorted(
    satellite_dir.glob("*.jp2")
)

band_records = []

for path in band_files:

    name = path.stem

    if "_B04_10m" in name:
        band = "B04"

    elif "_B08_10m" in name:
        band = "B08"

    else:
        continue

    parts = name.split("_")

    band_records.append(
        {
            "filename": path.name,
            "acquisition_date": parts[0],
            "satellite": parts[1],
            "tile": parts[2],
            "band": band,
            "size_mb": path.stat().st_size / (1024 ** 2),
            "path": str(path)
        }
    )

downloaded_bands = pd.DataFrame(band_records)

print(f"Total B04/B08 files: {len(downloaded_bands)}")
print()

display(
    downloaded_bands[
        [
            "acquisition_date",
            "satellite",
            "tile",
            "band",
            "size_mb"
        ]
    ]
)

### 10.2 Validate Complete B04 and B08 Acquisition Pairs [¶](#102-validate-complete-b04-and-b08-acquisition-pairs)


In [ ]:

##
## Current processing periods:
##   - November-February
##   - March-April
##
## May-June is intentionally excluded until all selected
## tile pairs are available.

from pathlib import Path
import pandas as pd
import rasterio


# ---------------------------------------------------------
# 1. Define the periods currently ready for processing
# ---------------------------------------------------------

periods_to_process = {
    "November-February": (
        pd.Timestamp("2025-11-01"),
        pd.Timestamp("2026-02-28")
    ),
    "March-April": (
        pd.Timestamp("2026-03-01"),
        pd.Timestamp("2026-04-30")
    ),
     "May-June": (
        pd.Timestamp("2026-05-01"),
        pd.Timestamp("2026-06-30")
    )
    
}


# ---------------------------------------------------------
# 2. Build the inventory from the downloaded files
# ---------------------------------------------------------

band_files = sorted(
    satellite_dir.glob("*.jp2")
)

records = []

for path in band_files:

    parts = path.stem.split("_")

    # Expected:
    # YYYY-MM-DD_Satellite_Tile_B04_10m

    if len(parts) < 5:
        continue

    acquisition_date = pd.Timestamp(parts[0])

    # Ignore files outside the current processing periods
    if not any(
        start <= acquisition_date <= end
        for start, end in periods_to_process.values()
    ):
        continue

    satellite = parts[1]
    tile = parts[2]

    if "_B04_10m" in path.stem:
        band = "B04"

    elif "_B08_10m" in path.stem:
        band = "B08"

    else:
        continue

    records.append(
        {
            "acquisition_date": acquisition_date,
            "satellite": satellite,
            "tile": tile,
            "band": band,
            "path": path
        }
    )


downloaded_bands = pd.DataFrame(records)


# ---------------------------------------------------------
# 3. Assign the analysis period
# ---------------------------------------------------------

def assign_period(date):

    for period, (start, end) in periods_to_process.items():

        if start <= date <= end:
            return period

    return None


downloaded_bands["period"] = (
    downloaded_bands["acquisition_date"]
    .apply(assign_period)
)


# ---------------------------------------------------------
# 4. Check the expected four files per acquisition
# ---------------------------------------------------------

pair_summary = (
    downloaded_bands
    .groupby(
        [
            "period",
            "acquisition_date",
            "satellite"
        ]
    )
    .agg(
        file_count=("path", "count"),
        tiles=("tile", "nunique"),
        bands=("band", "nunique")
    )
    .reset_index()
)


pair_summary["complete"] = (
    (pair_summary["file_count"] == 4)
    & (pair_summary["tiles"] == 2)
    & (pair_summary["bands"] == 2)
)


display(pair_summary)


# ---------------------------------------------------------
# 5. Report incomplete pairs
# ---------------------------------------------------------

incomplete = pair_summary[
    ~pair_summary["complete"]
]

if len(incomplete) == 0:

    print(
        "\nAll current November-February and March-April "
        "acquisition pairs are complete."
    )

else:

    print("\nIncomplete acquisition pairs:")
    display(incomplete)


# ---------------------------------------------------------
# 6. Validate raster structure
# ---------------------------------------------------------

validation_records = []

for _, row in downloaded_bands.iterrows():

    path = row["path"]

    with rasterio.open(path) as src:

        validation_records.append(
            {
                "acquisition_date": row["acquisition_date"],
                "satellite": row["satellite"],
                "tile": row["tile"],
                "band": row["band"],
                "crs": src.crs.to_string(),
                "width": src.width,
                "height": src.height,
                "resolution_x": src.res[0],
                "resolution_y": src.res[1],
                "count": src.count,
                "dtype": src.dtypes[0],
                "nodata": src.nodata
            }
        )


raster_validation = pd.DataFrame(
    validation_records
)


display(raster_validation)

## 11. Export Acquisition Tables and Figures for Reporting [¶](#11-export-acquisition-tables-and-figures-for-reporting)

Export the inventory, screening, selection, and download-validation tables that were produced in this acquisition workflow. These CSV files can be reused in the project report, PDF, or presentation. Any Matplotlib figures still open in the current session are saved with acquisition-specific filenames.

Only tables that exist and contain data in the current session are exported. This cell does not create new analyses or figures. Rerun it after updating the scene selection or downloading additional assets.


In [ ]:
from pathlib import Path
import pandas as pd

# Report-output directories
OUTPUTS_DIR = PROJECT_DIR / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Tables belonging to the acquisition and input-validation workflow.
table_names = [
    "scene_inventory",
    "tile_summary",
    "date_summary",
    "screening_summary",
    "candidate_dates",
    "inspect_dates",
    "candidate_ranked",
    "candidate_period_summary",
    "candidate_inventory",
    "duplicate_dates",
    "acquisition_pairs",
    "scl_inventory",
    "scl_validation",
    "aoi_scl_summary",
    "ndvi_candidates",
    "ndvi_selection",
    "selected_acquisitions",
    "downloaded_bands",
    "pair_summary",
    "raster_validation",
]

saved_tables = []

for table_name in table_names:
    table = globals().get(table_name)

    if isinstance(table, pd.DataFrame) and not table.empty:
        output_path = TABLES_DIR / f"sentinel2_{table_name}.csv"
        table.to_csv(output_path, index=False)
        saved_tables.append(output_path.name)

print(f"Tables saved: {len(saved_tables)}")
for filename in saved_tables:
    print(f"  {filename}")

# Save figures only when Matplotlib figures exist in this session.
try:
    import matplotlib.pyplot as plt

    figure_numbers = plt.get_fignums()

    for figure_number in figure_numbers:
        figure = plt.figure(figure_number)
        output_path = FIGURES_DIR / f"sentinel2_acquisition_figure_{figure_number:02d}.png"
        figure.savefig(
            output_path,
            dpi=300,
            bbox_inches="tight",
        )

    print(f"Figures saved: {len(figure_numbers)}")
except ImportError:
    print("Matplotlib is unavailable; no figures were saved.")


# Workflow Summary [¶](#12-workflow-summary)

This notebook discovers, selects, downloads, and validates Sentinel-2 Level-2A source imagery for the GeoSoil-Cuba Jaruco/Mayabeque area of interest (AOI). It prepares a documented set of source observations for the separate NDVI-processing notebook. **NDVI calculation and AOI-level raster production are intentionally outside the scope of this notebook.**

### Project setup and source data [¶](#121-project-setup-and-source-data)

- Set the project root using `PROJECT_DIR = Path("..").resolve()`.
- Loaded the curated soil–vegetation GeoPackage at `data/processed/vector/soil_vegetation_analysis.gpkg` as the authoritative AOI-related vector input.
- Kept original Sentinel-2 JP2 assets separate from derived raster products by storing them under `data/raw/satellite/sentinel2/`.

### Sentinel-2 observation inventory and selection [¶](#122-sentinel-2-observation-inventory-and-selection)

- Queried the Copernicus Data Space STAC catalogue for Sentinel-2 Level-2A observations over the study period.
- Organized observations by acquisition date, satellite platform, and MGRS tile, including tiles **17QLF** and **17QMF**.
- Used scene-level cloud-cover metadata for preliminary screening, then assessed SCL-derived conditions over the AOI to support acquisition selection.
- Selected temporally distributed candidate acquisitions for downstream NDVI analysis.

### SCL acquisition and validation [¶](#123-scl-acquisition-and-validation)

- Built an inventory of candidate Scene Classification Layer (SCL) assets.
- Downloaded the 20 m SCL rasters for the selected candidate observations.
- Checked download completeness, raster readability, structural consistency, and expected classification values.
- Calculated AOI-specific SCL statistics to distinguish scene-wide cloud metadata from conditions within the study area.

### B04 and B08 acquisition and validation [¶](#124-b04-and-b08-acquisition-and-validation)

- Downloaded Sentinel-2 B04 (red) and B08 (near-infrared) 10 m JP2 assets for selected acquisitions.
- Inventoried downloaded files by acquisition date, satellite, tile, and band.
- Checked whether the required B04/B08 pairs were complete and readable before they are used for NDVI calculation.
- Stored the original imagery in `data/raw/satellite/sentinel2/`.

### Report outputs [¶](#125-report-outputs)

- Exported available acquisition, screening, selection, and validation tables as CSV files under `outputs/tables/`, using the `sentinel2_` filename prefix.
- Saved any open Matplotlib figures under `outputs/figures/` with acquisition-specific filenames.
- Exported only tables and figures available in the current notebook session; rerun the export cell after changing the selection or download state.

### Stage 3 conclusions [¶](#126-overall-conclusions)

This notebook establishes the acquisition and input-validation stage of the Sentinel-2 workflow. Scene-wide cloud percentages are useful for preliminary prioritization, but AOI-specific SCL statistics provide more directly relevant evidence about observation conditions over the study area. Download and pair-validation checks help identify missing or unreadable inputs before raster processing begins.

The number of usable observations depends on the files successfully downloaded and validated during the current run. The output tables should therefore be regenerated after changes to the candidate selection or source files. A complete B04/B08 pair is a necessary input condition for NDVI calculation, but it does not by itself guarantee that every pixel is free of clouds or other contamination.

